# Run: teaching experiment (teacher vs stand-in players)

Runs the OOD experiments E1..E5 from `ood_testing.ipynb` and the balanced-outcomes metric from
`metrics.ipynb`: build a history for a persona, then play the teacher against the same / different
persona and record model estimates, OOD scores, fallback rate and closeness.

**TODO checklist**
- [ ] Implement styled personas and `weakness_target_score` first (otherwise every decision is a fallback or fails).
- [ ] Fix N games per experiment and the balanced-outcomes target before running.
- [ ] Save results (CSV) and plots for the Wiki.

In [ ]:
%run ./ood_testing.ipynb

In [ ]:
import pandas as pd

N_HISTORY, N_TEST = 10, 20
analyst = make_engine(prefer_uci=True)                  # Stockfish if installed
player_engine = make_engine(prefer_uci=True, elo=ENGINE_CFG["default_elo"])

## E1: consistent persona

In [ ]:
persona = PersonaPlayer(PERSONAS["solid"], make_engine(prefer_uci=True, elo=1700), seed=0)
model = PlayerModel("E1")
history = [play_persona_game(persona, _EnginePolicy(player_engine), game_id="E1-hist") for _ in range(N_HISTORY)]
hist_rows = evaluate_model_on_games(model, history, analyst, chess.WHITE, true_elo=1700)

teacher = TeacherPolicy(player_engine, analyst, model)
outcomes, rows = [], []
for i in range(N_TEST):
    g = play_persona_game(persona, teacher, game_id=f"E1-{i}")
    # TODO: feed teacher.observe_human_move during the game for the live OOD check
    b = g.board(); evals = []
    for mv in g.mainline_moves():
        b.push(mv)
        if not b.is_game_over():
            c = analyst.analyse(b)[0]
            evals.append(c.score_cp if b.turn == chess.WHITE else -c.score_cp)
    outcomes.append(GameOutcome(g.headers["Result"], evals, "black", f"E1-{i}"))
print(balanced_share(outcomes))
summarize_games(outcomes)

## E2..E5 (TODO: same structure with a different test persona)